### Set Up

In [1]:
import os
#os.environ["KERAS_BACKEND"] = "torch"

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import keras
import torch
import tensorflow as tf
import math
import re
import collections
from typing import Dict, List, Tuple
import os, pathlib, shutil, random
import string
import keras_hub
import kagglehub
from functools import partial
import json

In [3]:
# Quick health check to prove it works:
print("Is PyTorch utilizing the GPU?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

Is PyTorch utilizing the GPU?: True
Device Name: Tesla T4


### Kaggle Setup

In [4]:
#kagglehub.login() # KGAT_7c8a631bcd7262074bb82f28abfd6ebe

### Load Model

In [5]:
#gemma_lm = keras_hub.models.CausalLM.from_preset("gemma3_1b", dtype="float32")
#gemma_lm.summary()

In [6]:
!tar -xvf /content/drive/MyDrive/gemma3-keras-gemma3_1b-v3.tar.gz

assets/tokenizer/
assets/
assets/tokenizer/vocabulary.spm
config.json
metadata.json
model.weights.h5
preprocessor.json
task.json
tokenizer.json


In [7]:
gemma_lm = keras_hub.models.CausalLM.from_preset("/content", dtype="float32")
gemma_lm.summary()

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 1152)        │     999,885,952 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     301,989,888 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 999,885,952 (3.72 GB)

 Trainable params: 999,885,952 (3.72 GB)

 Non-trainable params: 0 (0.00 B)

### Inference

In [8]:
gemma_lm.compile(sampler="greedy")

In [9]:
gemma_lm.generate("A piece of advice", max_length=100)

'A piece of advice from a former student of mine:\n\n<blockquote>“I’m not sure if you’ve heard of it, but I’ve been told that the best way to learn a language is to speak it. I’ve been told that if you speak it, you’ll be able to understand it better. I’ve been told that if you understand it better, you’ll be able to speak it better. I’ve been told that if you speak it'

In [10]:
gemma_lm.generate("How can I make brownies?", max_length=100)

"How can I make brownies?\n\n[User 0001]\n\nI'm trying to make brownies for my son's birthday party. I've never made brownies before. I've seen a lot of recipes on the internet, but I'm not sure which one to use. I'm not sure if I should use a box mix or make my own. I'm not sure if I should use a brownie mix or a brownie mix. I'm"

In [11]:
gemma_lm.generate("The following brownie recipe is easy to make in just a few steps.\n\nYou can start by", max_length=100)

'The following brownie recipe is easy to make in just a few steps.\n\nYou can start by melting the butter and sugar in a saucepan over medium heat.\n\nThen add the eggs and vanilla extract and mix well.\n\nNext, add the flour and baking powder and mix well.\n\nFinally, add the chocolate chips and mix well.\n\nThe brownies are ready to be baked in a preheated oven at 350 degrees Fahrenheit for 20 minutes.\n\nThe brownies are ready to'

In [12]:
gemma_lm.generate("Tell me about the 542nd president of the United States.", max_length=40)

'Tell me about the 542nd president of the United States.\n\nThe 542nd president of the United States was James A. Garfield.\n\nThe 542'

In [13]:
gemma_lm.generate("Hi", max_length=40)

'Hi,\n\nI have a problem with my 2008 328i. I have a problem with the car not starting. I have tried to start it with the key'

### Load data for Finetuning

In [14]:
PROMPT_TEMPLATE = """[instruction]\n{}[end]\n[response]\n"""
RESPONSE_TEMPLATE = """{}[end]"""

dataset_path = keras.utils.get_file(origin=("https://hf.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl"))
data = {"prompts": [], "responses": []}
with open(dataset_path) as file:
    for line in file:
        features = json.loads(line)
        if features["context"]:
            continue
        data["prompts"].append(PROMPT_TEMPLATE.format(features["instruction"]))
        data["responses"].append(RESPONSE_TEMPLATE.format(features["response"]))



13085339/13085339 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [15]:
i = 10
print(data["prompts"][i])

[instruction]
Give me the top 5 golf equipment company names.[end]
[response]



In [16]:
print(data["responses"][i])

Titleist, Taylormade, Callaway, Ping, Cobra[end]


### Preprocessing

In [17]:
ds = tf.data.Dataset.from_tensor_slices(data).shuffle(2000).batch(2)
val_ds = ds.take(100)
train_ds = ds.skip(100)

In [18]:
preprocessor = gemma_lm.preprocessor
preprocessor.sequence_length = 512

In [19]:
batch = next(iter(train_ds))
x, y, sample_weight = preprocessor(batch)

In [20]:
x["token_ids"]

<tf.Tensor: shape=(2, 512), dtype=int32, numpy=
array([[     2, 236840,  22768, ...,      0,      0,      0],
       [     2, 236840,  22768, ...,      0,      0,      0]], dtype=int32)>

In [21]:
x["padding_mask"]

<tf.Tensor: shape=(2, 512), dtype=bool, numpy=
array([[ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False]])>

In [22]:
y

<tf.Tensor: shape=(2, 512), dtype=int32, numpy=
array([[236840,  22768, 236842, ...,      0,      0,      0],
       [236840,  22768, 236842, ...,      0,      0,      0]], dtype=int32)>

In [23]:
sample_weight

<tf.Tensor: shape=(2, 512), dtype=bool, numpy=
array([[False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]])>

In [24]:
x["token_ids"][0, :5], y[0, :5]

(<tf.Tensor: shape=(5,), dtype=int32, numpy=array([     2, 236840,  22768, 236842,    107], dtype=int32)>,
 <tf.Tensor: shape=(5,), dtype=int32, numpy=array([236840,  22768, 236842,    107,  15938], dtype=int32)>)

### LoRA

In [25]:
# Enabling LoRA training for a KerasHub model
gemma_lm.backbone.enable_lora(rank=4)

In [26]:
gemma_lm.summary()

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 1152)        │   1,000,538,240 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     301,989,888 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 1,000,538,240 (3.73 GB)

 Trainable params: 652,288 (2.49 MB)

 Non-trainable params: 999,885,952 (3.72 GB)

In [27]:
gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(5e-5),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

gemma_lm.fit(train_ds, validation_data=val_ds, epochs=1)

5172/5172 ━━━━━━━━━━━━━━━━━━━━ 6429s 1s/step - loss: 0.3263 - sparse_categorical_accuracy: 0.5426 - val_loss: 0.3090 - val_sparse_categorical_accuracy: 0.5374


### Inference after FineTune (LoRA Fine Tuned Model)

In [31]:
print(gemma_lm.generate("[instruction]\nHow can I make brownies?[end]\n[response]\n", max_length=512,))

[instruction]
How can I make brownies?[end]
[response]
Brownies are a classic American dessert. They are made with a base of flour, sugar, and eggs, and then topped with chocolate or other sweet toppings. There are many variations of brownies, including chocolate chip, peanut butter, and caramel. You can also add other ingredients like nuts, chocolate chips, or chocolate chips to the brownie batter. You can also add other ingredients like nuts, chocolate chips, or chocolate chips to the brownie batter. You can also add other ingredients like nuts, chocolate chips, or chocolate chips to the brownie batter. You can also add other ingredients like nuts, chocolate chips, or chocolate chips to the brownie batter. You can also add other ingredients like nuts, chocolate chips, or chocolate chips to the brownie batter. You can also add other ingredients like nuts, chocolate chips, or chocolate chips to the brownie batter. You can also add other ingredients like nuts, chocolate chips, or chocol

In [32]:
print(gemma_lm.generate("[instruction]\nTell me about the 542nd president of the United States.[end]\n[response]\n", max_length=512,))

[instruction]
Tell me about the 542nd president of the United States.[end]
[response]
The 542nd president of the United States was James A. Garfield. He was born on January 19, 1831 in New York City. He was the 20th president of the United States. He was assassinated on July 2, 1881 in Washington, D.C. He was the 20th president of the United States. He was assassinated by Charles J. Guiteau, a former employee of the U.S. Treasury Department. Garfield was a lawyer and a politician. He was a member of the Democratic Party. He was the 20th president of the United States. He was assassinated by Charles J. Guiteau, a former employee of the U.S. Treasury Department. Garfield was born on January 19, 1831 in New York City. He was the 20th president of the United States. He was the 20th president of the United States. He was assassinated on July 2, 1881 in Washington, D.C. He was the 20th president of the United States. He was assassinated by Charles J. Guiteau, a former employee of the U.S. Tr

In [33]:
print(gemma_lm.generate("[instruction]\nHi[end]\n[response]\n", max_length=512,))

[instruction]
Hi[end]
[response]
Hello[end]
